# Main Rerun Stage 2 Final Model SHAP

这份 notebook 只做一件事：

- 对 `main_rerun` 当前正式 `Stage 2` 最优模型做一次 full-train refit
- 在 `primary_holdout` 上输出一张 SHAP summary 图

固定口径：

- 只做 `Stage 2`
- 只做 `selected final model`
- 不重跑 nested CV
- 不依赖外部脚本
- 只读取 `main_rerun/` 内已有 artifact


## Inputs

这份 notebook 只读取以下 3 个输入：

- `main_rerun/artifacts/stage2/stage2_daily_features_long_v2_main_rerun.csv`
- `main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_selected_model.csv`
- `main_rerun/artifacts/stage2/final_compare/stage2_nested_cv_model_compare_holdout_metrics.csv`

输出目录固定为：

- `main_rerun/artifacts/stage2/final_compare/shap/`

运行时请切到项目内 `.venv` 对应的 interpreter / kernel。
当前系统默认 `python3` kernel 指向的是 `anaconda`，里面没有 `shap`。
项目内 `.venv` 已经带了 `shap`，直接用它最省事。


In [ ]:
from pathlib import Path
import json
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import shap
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires the `shap` package. Install it in the active environment before running."
    ) from exc

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

CWD = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [CWD, *CWD.parents]:
    if (candidate / "main_rerun" / "artifacts" / "stage2").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing `main_rerun/artifacts/stage2`.")

os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".mpl-cache"))

MAIN_RERUN_ROOT = PROJECT_ROOT / "main_rerun"
STAGE2_ARTIFACT_DIR = MAIN_RERUN_ROOT / "artifacts" / "stage2"
FINAL_COMPARE_DIR = STAGE2_ARTIFACT_DIR / "final_compare"
OUT_DIR = FINAL_COMPARE_DIR / "shap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = STAGE2_ARTIFACT_DIR / "stage2_daily_features_long_v2_main_rerun.csv"
SELECTED_MODEL_PATH = FINAL_COMPARE_DIR / "stage2_nested_cv_model_compare_selected_model.csv"
HOLDOUT_METRICS_PATH = FINAL_COMPARE_DIR / "stage2_nested_cv_model_compare_holdout_metrics.csv"

TARGET_COL = "gold_ret_1d"
DATE_COL = "date"
TRAIN_LABEL = "train_pool"
HOLDOUT_LABEL = "primary_holdout"
RANDOM_STATE = 42

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STAGE2_ARTIFACT_DIR:", STAGE2_ARTIFACT_DIR)
print("FINAL_COMPARE_DIR:", FINAL_COMPARE_DIR)
print("OUT_DIR:", OUT_DIR)


In [ ]:
P_TACO_V2_CORE_FEATURES = [
    "p_taco_any",
    "p_taco_top1",
    "p_taco_tail_010",
    "p_taco_any_decay_2d",
]

CORE_CONTROL_FEATURES = [
    "dxy_ret_1d_lag1",
    "real_yield_change_5d_lag1",
    "vix_ma_5d_lag1",
    "vix_change_5d_lag1",
    "spx_ret_1d_lag1",
    "spx_drawdown_5d_lag1",
    "gold_spx_corr_20d_lag1",
]

MODEL_SPEC_CATALOG = {
    "two_stage_v2_core": P_TACO_V2_CORE_FEATURES + CORE_CONTROL_FEATURES,
}


def parse_best_param_string(text):
    data = json.loads(text)
    return {key.replace("model__", ""): value for key, value in data.items()}


def coerce_matrix(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


def coerce_shap_values(shap_output):
    if hasattr(shap_output, "values"):
        values = shap_output.values
    else:
        values = shap_output
    if isinstance(values, list):
        values = values[-1]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    return values


def build_pipeline(features, model_params):
    return Pipeline(
        steps=[
            (
                "preprocessor",
                ColumnTransformer(
                    transformers=[
                        (
                            "num",
                            Pipeline(
                                steps=[
                                    ("imputer", SimpleImputer(strategy="median")),
                                    ("scaler", StandardScaler()),
                                ]
                            ),
                            features,
                        )
                    ],
                    remainder="drop",
                ),
            ),
            (
                "model",
                RandomForestRegressor(
                    random_state=RANDOM_STATE,
                    n_jobs=1,
                    **model_params,
                ),
            ),
        ]
    )


def save_summary_plot(shap_values, feature_matrix, feature_names, out_path):
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_values,
        feature_matrix,
        feature_names=feature_names,
        show=False,
        max_display=len(feature_names),
    )
    plt.title("Main Rerun Stage 2 Final Model SHAP Summary")
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()


In [ ]:
selected_model_df = pd.read_csv(SELECTED_MODEL_PATH)
if len(selected_model_df) != 1:
    raise ValueError(
        f"Expected exactly one selected Stage 2 final model row, got {len(selected_model_df)}."
    )

selected_row = selected_model_df.iloc[0]
variant_key = selected_row["variant_key"]
model_name = selected_row["model_name"]
spec_name = selected_row["spec_name"]

if model_name != "random_forest":
    raise ValueError(f"Selected model is `{model_name}`, not `random_forest`.")
if spec_name not in MODEL_SPEC_CATALOG:
    raise ValueError(f"Unsupported spec_name `{spec_name}`.")

feature_cols = MODEL_SPEC_CATALOG[spec_name]

holdout_metrics_df = pd.read_csv(HOLDOUT_METRICS_PATH)
selected_holdout_row = holdout_metrics_df[
    (holdout_metrics_df["variant_key"] == variant_key)
    & (holdout_metrics_df["model_name"] == model_name)
    & (holdout_metrics_df["spec_name"] == spec_name)
].copy()
if len(selected_holdout_row) != 1:
    raise ValueError("Expected exactly one selected holdout metrics row for final model.")
selected_holdout_row = selected_holdout_row.iloc[0]

model_params = parse_best_param_string(selected_holdout_row["best_params_json"])
selected_holdout_row[["variant_key", "model_name", "spec_name", "best_params_json", "rmse", "mae", "oos_r2", "sign_accuracy"]]


In [ ]:
with (OUT_DIR / "stage2_final_selected_model_params.json").open("w", encoding="utf-8") as handle:
    json.dump(model_params, handle, indent=2, ensure_ascii=False)

df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.normalize()
df = df[
    (df["variant_key"] == variant_key)
    & (df["stage2_split"].isin([TRAIN_LABEL, HOLDOUT_LABEL]))
].copy()
df = df.dropna(subset=[TARGET_COL]).sort_values(DATE_COL).reset_index(drop=True)

missing_features = [col for col in feature_cols if col not in df.columns]
if missing_features:
    raise ValueError(f"Missing required features: {missing_features}")

train_df = df[df["stage2_split"] == TRAIN_LABEL].copy().reset_index(drop=True)
holdout_df = df[df["stage2_split"] == HOLDOUT_LABEL].copy().reset_index(drop=True)
if train_df.empty or holdout_df.empty:
    raise ValueError("Train pool or holdout slice is empty; cannot fit SHAP model.")

print("variant_key:", variant_key)
print("model_name:", model_name)
print("spec_name:", spec_name)
print("train_pool rows:", len(train_df))
print("primary_holdout rows:", len(holdout_df))
print("feature count:", len(feature_cols))


In [ ]:
pipeline = build_pipeline(features=feature_cols, model_params=model_params)
pipeline.fit(train_df[feature_cols], train_df[TARGET_COL])

holdout_pred = pipeline.predict(holdout_df[feature_cols])
holdout_prediction_df = holdout_df[[DATE_COL, TARGET_COL]].copy()
holdout_prediction_df["variant_key"] = variant_key
holdout_prediction_df["model_name"] = model_name
holdout_prediction_df["spec_name"] = spec_name
holdout_prediction_df["y_pred"] = holdout_pred
holdout_prediction_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_predictions.csv", index=False)

holdout_prediction_df.head()


In [ ]:
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]
holdout_matrix = coerce_matrix(preprocessor.transform(holdout_df[feature_cols]))

explainer = shap.TreeExplainer(model)
shap_values = coerce_shap_values(explainer.shap_values(holdout_matrix))

summary_plot_path = OUT_DIR / "stage2_final_selected_model_holdout_shap_summary.png"
save_summary_plot(
    shap_values=shap_values,
    feature_matrix=holdout_matrix,
    feature_names=feature_cols,
    out_path=summary_plot_path,
)

print("Saved summary plot to:", summary_plot_path)


In [ ]:
shap_values_df = pd.DataFrame(shap_values, columns=feature_cols)
shap_values_df.insert(0, DATE_COL, holdout_df[DATE_COL].dt.strftime("%Y-%m-%d").to_numpy())
shap_values_df.insert(1, TARGET_COL, holdout_df[TARGET_COL].to_numpy())
shap_values_df.insert(2, "y_pred", holdout_pred)
shap_values_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_shap_values.csv", index=False)

feature_value_df = holdout_df[[DATE_COL] + feature_cols].copy()
feature_value_df[DATE_COL] = feature_value_df[DATE_COL].dt.strftime("%Y-%m-%d")
feature_value_df.to_csv(OUT_DIR / "stage2_final_selected_model_holdout_feature_values.csv", index=False)

shap_importance_df = (
    pd.DataFrame(
        {
            "feature_name": feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "mean_shap": shap_values.mean(axis=0),
            "feature_value_mean_holdout": holdout_df[feature_cols].mean().to_numpy(),
            "feature_value_std_holdout": holdout_df[feature_cols].std(ddof=0).to_numpy(),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance_df.to_csv(OUT_DIR / "stage2_final_selected_model_shap_importance.csv", index=False)

shap_importance_df


In [ ]:
summary_lines = [
    "# Main Rerun Stage 2 Final Model SHAP Summary",
    "",
    "## Final model",
    f"- `variant_key = {variant_key}`",
    f"- `model_name = {model_name}`",
    f"- `spec_name = {spec_name}`",
    f"- `best_params_json = {selected_holdout_row['best_params_json']}`",
    "",
    "## Sample",
    f"- `train_pool rows = {len(train_df)}`",
    f"- `primary_holdout rows = {len(holdout_df)}`",
    f"- `feature count = {len(feature_cols)}`",
    "",
    "## Holdout metrics",
    f"- `RMSE = {selected_holdout_row['rmse']:.6f}`",
    f"- `MAE = {selected_holdout_row['mae']:.6f}`",
    f"- `OOS R2 = {selected_holdout_row['oos_r2']:.6f}`",
    f"- `sign_accuracy = {selected_holdout_row['sign_accuracy']:.6f}`",
    "",
    "## Top Mean |SHAP| Features",
]

for _, row in shap_importance_df.head(10).iterrows():
    summary_lines.append(
        f"- `{row['feature_name']}`: `mean|SHAP| = {row['mean_abs_shap']:.6f}`, "
        f"`mean SHAP = {row['mean_shap']:.6f}`"
    )

summary_lines.extend(
    [
        "",
        "## Artifacts",
        f"- `summary_plot = {summary_plot_path.relative_to(PROJECT_ROOT)}`",
        f"- `shap_importance = {(OUT_DIR / 'stage2_final_selected_model_shap_importance.csv').relative_to(PROJECT_ROOT)}`",
        f"- `shap_values = {(OUT_DIR / 'stage2_final_selected_model_holdout_shap_values.csv').relative_to(PROJECT_ROOT)}`",
        f"- `feature_values = {(OUT_DIR / 'stage2_final_selected_model_holdout_feature_values.csv').relative_to(PROJECT_ROOT)}`",
        f"- `holdout_predictions = {(OUT_DIR / 'stage2_final_selected_model_holdout_predictions.csv').relative_to(PROJECT_ROOT)}`",
    ]
)

summary_path = OUT_DIR / "stage2_final_selected_model_shap_summary.md"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

print("Saved markdown summary to:", summary_path)
print()
print("\n".join(summary_lines))
